In [17]:
import duckdb
import pandas as pd
import json

In [2]:
df_epci = pd.read_csv("../data/processed/epci_membres.csv")

In [13]:
df_mediation_num = pd.read_csv("../data/data_medi_num/raw/mediation_numerique.csv",low_memory=False)
df_mediation_num

,id,pivot,nom,commune,code_postal,code_insee,adresse,complement_adresse,latitude,longitude,...,publics_specifiquement_adresses,prise_en_charge_specifique,frais_a_charge,dispositif_programmes_nationaux,formations_labels,autres_formations_labels,modalites_acces,modalites_accompagnement,fiche_acces_libre,prise_rdv
0,Coop-numérique_201a8c38-32ba-4332-99fa-8888a65...,00000000000000,Espace de Vie Sociale,Arras,62000,62041,8 Place Mere Teresa,NaN,50.303410,2.739115,...,NaN,NaN,NaN,Conseillers numériques,NaN,QPV,NaN,NaN,NaN,NaN
1,Coop-numérique_0bf02833-eb32-4e74-8211-905b63c...,20007136300010,CA VICHY COMMUNAUTE,Vichy,3200,03310,9 Place Charles de Gaulle,NaN,46.124577,3.426321,...,NaN,NaN,Gratuit,Conseillers numériques,NaN,NaN,NaN,Dans un atelier collectif|Accompagnement indiv...,NaN,NaN
2,Coop-numérique_391062d5-31f3-479e-bc14-57649fc...,00000000000000,France services,La Motte-du-Caire,4250,04134,MAISON DE PAYS,NaN,44.341152,6.027605,...,NaN,NaN,Gratuit,Conseillers numériques,NaN,ZRR,NaN,NaN,NaN,NaN
3,France-Services_3028,00000000000000,Bus France services de Vendôme,Vendôme,41100,41269,Rue des Maillettes,NaN,47.809537,1.059398,...,NaN,NaN,Gratuit,France Services,NaN,QPV,Téléphoner|Contacter par mail,NaN,NaN,NaN
4,Coop-numérique_571bcfa7-36d8-4d7f-8f90-77d0c79...,00000000000000,Médiathèque Jacques Baumel,Rueil-Malmaison,92500,92063,15 Boulevard du Maréchal Foch,NaN,48.877941,2.180088,...,NaN,NaN,NaN,Conseillers numériques,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17211,Coop-numérique_b51a6072-de0d-4d1c-96e0-9b85fa9...,00000000000000,PMS de Périphérie et Vallée,Thonon-les-Bains,74200,74281,2 Avenue du Vernay,NaN,46.363973,6.461930,...,NaN,NaN,NaN,Conseillers numériques,NaN,NaN,NaN,NaN,NaN,NaN
17212,Coop-numérique_8916c35b-80f9-4d46-bd43-391aa2a...,00000000000000,Mairie de Rouessé-Vassé,Rouessé-Vassé,72140,72255,3 Place de la Mairie,NaN,48.159829,-0.198405,...,NaN,NaN,NaN,Conseillers numériques,NaN,ZRR,NaN,NaN,NaN,NaN
17213,Coop-numérique_d1caac1d-9e21-480e-bff8-a4b7245...,21790176800010,Campus de projets de Ménigoute,Ménigoute,79340,79176,Rue de Saint-Maixent,NaN,46.493141,-0.058340,...,NaN,NaN,Gratuit,Conseillers numériques,NaN,ZRR,NaN,NaN,NaN,NaN
17214,Coop-numérique_b079f89b-a56f-4838-84ff-50593e0...,00000000000000,5 rue des Ecoles - 67680 EPFIG,Epfig,67680,67125,5 Rue des Ecoles,NaN,48.360219,7.463530,...,NaN,NaN,NaN,Conseillers numériques,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
df_mediation_num['code_postal'] = df_mediation_num['code_postal'].astype(str).str.zfill(5)
df_med_num = df_mediation_num[['commune', 'code_postal', 'code_insee','adresse']].sort_values(by='code_insee')  
df_med_num = df_med_num.drop_duplicates()


In [31]:
df_med_num

,commune,code_postal,code_insee,adresse
7500,L'Abergement-Clémenciat,01400,01001,119 Route de la Fontaine
8195,L'Abergement-de-Varey,01640,01002,null LE VILLAGE
2175,Ambérieu-en-Bugey,01500,01004,5 Rue Berthelot
1453,Ambérieu-en-Bugey,01500,01004,Place Jules Ferry
14611,Ambérieux-en-Dombes,01330,01005,289 Rue Gombette
...,...,...,...,...
15295,LA-TARDIERE,85120,NaN,1 Rue Augustin de Hargues
15385,Le-Merlerault,61240,NaN,PLACE DE L'HÔTEL DE VILLE
15495,CAMON,80334,NaN,26 Place du Général Leclerc
16038,HELLEMMES-LILLE,59260,NaN,5 rue Jean Raymond Degrève


In [35]:
df_isna = df_med_num[df_med_num['code_insee'].isna()].sort_values(by='commune')
df_isna

,commune,code_postal,code_insee,adresse
6497,AIX-LA-DURANNE,13100,NaN,forum georges charpak
10019,BLETTRANS,39140,NaN,3 Place d'orion
14407,Bordères-et-Lamensens,40270,NaN,168 Chemin de pébon
15495,CAMON,80334,NaN,26 Place du Général Leclerc
1390,CAPAVENIR-VOSGES,88150,NaN,6 AV DES FUSILLES
...,...,...,...,...
6829,THOUARSAIS-BOUILDROUX,85410,NaN,54 Rue du centre
9384,TOURETTES-SUR-LOUP,06140,NaN,place maximin escalier
15025,VALENCIENES,59300,NaN,10 rue des Ursulines
10293,VERDUN-SUR-LE-DOUBS,71350,NaN,16 Rue de la République


In [38]:
df_com = pd.read_csv("../data/raw/communes_france_2025.csv",low_memory=False)

In [39]:
df_isna['dep_code'] = df_isna['code_postal'].astype(str).str.zfill(5).str[:2]

def find_code_insee(row):
    com = str(row['commune']).upper()
    dep = row['dep_code']
    matches = df_com[
        (df_com['dep_code'] == dep) &
        (df_com['nom_standard_majuscule'].str.contains(com, case=False, na=False, regex=False))
    ]
    return matches['code_insee'].iloc[0] if not matches.empty else None

df_isna['code_insee'] = df_isna.apply(find_code_insee, axis=1)
df_isna

,commune,code_postal,code_insee,adresse,dep_code
6497,AIX-LA-DURANNE,13100,None,forum georges charpak,13
10019,BLETTRANS,39140,None,3 Place d'orion,39
14407,Bordères-et-Lamensens,40270,None,168 Chemin de pébon,40
15495,CAMON,80334,80164,26 Place du Général Leclerc,80
1390,CAPAVENIR-VOSGES,88150,None,6 AV DES FUSILLES,88
...,...,...,...,...,...
6829,THOUARSAIS-BOUILDROUX,85410,None,54 Rue du centre,85
9384,TOURETTES-SUR-LOUP,06140,None,place maximin escalier,06
15025,VALENCIENES,59300,None,10 rue des Ursulines,59
10293,VERDUN-SUR-LE-DOUBS,71350,71566,16 Rue de la République,71


In [40]:
#on enlève les missing value de df_isna et on les ajoute à df_med_num
df_isna_null = df_isna[df_isna['code_insee'].isna()]
df_isna_null

,commune,code_postal,code_insee,adresse,dep_code
6497,AIX-LA-DURANNE,13100,None,forum georges charpak,13
10019,BLETTRANS,39140,None,3 Place d'orion,39
14407,Bordères-et-Lamensens,40270,None,168 Chemin de pébon,40
1390,CAPAVENIR-VOSGES,88150,None,6 AV DES FUSILLES,88
4593,CEZAIS,85410,None,40 Rue du noyer,85
8266,CHERVES-RICHEMONT,16370,None,2 Place du Champ de Foire,16
7281,COSNE-SUR-LOIRE,58200,None,7bis Rue Eugène Pelletan,58
14183,COSNE-SUR-LOIRE,58200,None,40 Rue des Rivières Saint-Agnan,58
2388,ETRAT,42580,None,3 place de l'Eglise,42
8754,Eryaud-Crempse-Maurens,24140,None,Le bourg centre,24


In [41]:
df_com.loc[df_com['nom_standard'].str.contains("merlera", case=False, na=False)]

,Unnamed: 0,code_insee,nom_standard,nom_sans_pronom,nom_a,nom_de,nom_sans_accent,nom_standard_majuscule,typecom,typecom_texte,...,longitude_mairie,latitude_centre,longitude_centre,grille_densite,grille_densite_texte,niveau_equipements_services,niveau_equipements_services_texte,gentile,url_wikipedia,url_villedereve
23315,23315,61275,Le Merlerault,Merlerault,au Merlerault,du Merlerault,le-merlerault,LE MERLERAULT,COM,commune,...,0.284,48.701,0.28,6,Rural à habitat dispersé,1.0,centres locaux d'équipements et de services,Merluriens,https://fr.wikipedia.org/wiki/fr:Le Merlerault,https://villedereve.fr/ville/61275-le-merlerault


In [42]:

# Crée un dictionnaire avec les communes comme clés et des valeurs vides
mapping_com = {com: "" for com in df_isna_null['commune']}

In [52]:
mapping_com

{'AIX-LA-DURANNE': '13001',
 'BLETTRANS': '39056',
 'Bordères-et-Lamensens': '40049',
 'CAPAVENIR-VOSGES': '88465',
 'CEZAIS': '85292',
 'CHERVES-RICHEMONT': '16097',
 'COSNE-SUR-LOIRE': '58086',
 'ETRAT': '42092',
 'Eryaud-Crempse-Maurens': '24259',
 'Etroeugnt': '59218',
 'HELLEMMES---LILLE': '59350',
 'HELLEMMES-LILLE': '59350',
 'LA-TARDIERE': '85289',
 'LE-BÉNY-BOCAGE': '14061',
 'La-Chapelle-aux-Pots': '61275',
 'Le-Merlerault': '60333',
 'MARDYCK': '59183',
 'MONTIGNY-PRES-LOUHANS': '71303',
 'MORET-SUR-LOING': '77316',
 'MOREZ': '39368',
 'NANTEUIL-LE-HAUDOIN': '60446',
 'NUIT-SAINT-GEORGES': '21464',
 'Neussargues-en-Pinatelle': '15141',
 'PONT-A-BUCY': '02559',
 'PONT-DU-LOUP': '06148',
 'Richerbourg': '62706',
 'SAINT-MACOUX': '86247',
 'SAINT-SAVIOL': '86247',
 'SAINT-SULPICE-DE-COGNAC': '16097',
 'SECHELLES': '02004',
 'SENNECEY-SUR-SAÔNE-SAINT-ALBIN': '70482',
 'Saint-Macoux': '86247',
 'Saint-Paul-lez-Durance': '13099',
 'Saint-Saviol': '86247',
 'THOUARSAIS-BOUILDROUX':

In [43]:
valeurs_mapping = ['13001',
                   '39056',
                   '40049',
                   '88465',
                   '85292',
                   '16097',
                   '58086',
                   '42092',
                   '24259',
                   '59218',
                   '59350',
                   '59350',
                   '85289',
                   '14061',
                   '61275',
                   '60333',
                   '59183',
                   '71303',
                   '77316',
                   '39368',
                   '60446',
                    '21464',
                    '15141',
                    '02559',
                    '06148',
                   '62706',
                   '86247',
                   '86247',
                   '16097',
                   '02004',
                   '70482',
                   '86247',
                   '13099',
                   '86247',
                   '85292',
                   '06148',
                   '59606',
                   '59309'
                   ]

In [44]:
for i in range(len(valeurs_mapping)):
    mapping_com.update({list(mapping_com.keys())[i]: valeurs_mapping[i]})

In [45]:
df_isna_null['code_insee'] = df_isna_null['commune'].map(mapping_com)

/tmp/ipykernel_7578/2568894341.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_isna_null['code_insee'] = df_isna_null['commune'].map(mapping_com)


In [46]:
df_isna_null

,commune,code_postal,code_insee,adresse,dep_code
6497,AIX-LA-DURANNE,13100,13001,forum georges charpak,13
10019,BLETTRANS,39140,39056,3 Place d'orion,39
14407,Bordères-et-Lamensens,40270,40049,168 Chemin de pébon,40
1390,CAPAVENIR-VOSGES,88150,88465,6 AV DES FUSILLES,88
4593,CEZAIS,85410,85292,40 Rue du noyer,85
8266,CHERVES-RICHEMONT,16370,16097,2 Place du Champ de Foire,16
7281,COSNE-SUR-LOIRE,58200,58086,7bis Rue Eugène Pelletan,58
14183,COSNE-SUR-LOIRE,58200,58086,40 Rue des Rivières Saint-Agnan,58
2388,ETRAT,42580,42092,3 place de l'Eglise,42
8754,Eryaud-Crempse-Maurens,24140,24259,Le bourg centre,24


In [47]:
df_isna_not_null = df_isna[df_isna['code_insee'].notna()]

In [48]:
df_med_num_not_null = df_med_num[df_med_num['code_insee'].notna()]

In [49]:
df_med_num_not_null[df_med_num_not_null['code_insee'].str.startswith('13055')]

,commune,code_postal,code_insee,adresse
14669,Marseille,13004,13055,15 boulevard léglize
12880,Marseille,13007,13055,7 Rue sauveur tobelem
5182,Marseille,13010,13055,221 Avenue de la capelette
14893,Marseille,13002,13055,15 rue du terras
4362,Marseille,13015,13055,28 Boulevard de la padouane
...,...,...,...,...
4768,Marseille,13001,13055,18 rue colbert
14608,Marseille,13016,13055,46 Boulevard fenouil
8561,Marseille,13013,13055,15 Chemin des jonquilles
520,Marseille,13002,13055,1 place francis chirat


In [50]:
#on recolle les trois dataframes : df_isna_not_null, df_isna_null et df_med_num_not_null

df_med_num_final = pd.concat([df_isna_not_null, df_isna_null, df_med_num_not_null], ignore_index=True).sort_values(by='code_insee')
df_med_num_final

,commune,code_postal,code_insee,adresse,dep_code
66,L'Abergement-Clémenciat,01400,01001,119 Route de la Fontaine,NaN
67,L'Abergement-de-Varey,01640,01002,null LE VILLAGE,NaN
68,Ambérieu-en-Bugey,01500,01004,5 Rue Berthelot,NaN
69,Ambérieu-en-Bugey,01500,01004,Place Jules Ferry,NaN
70,Ambérieux-en-Dombes,01330,01005,289 Rue Gombette,NaN
...,...,...,...,...,...
15420,Saint-Martin,97150,97801,6 rue Léopold Mingau,NaN
15421,Saint-Martin,97150,97801,ANCIENNE ECOLE EVELYNA HALLEY,NaN
15422,Saint-Martin,97150,97801,3 IMPASSE CLAMY CHERRY,NaN
15423,Saint-Martin,97150,97801,Bat 4 Apt 401-402 Quartier d’Orléans,NaN


In [51]:
query = """ 
SELECT 
    count(*) AS nb_mediation,
    code_insee
FROM df_med_num_final
GROUP BY code_insee
ORDER BY code_insee
"""

df_mediation_num_grouped = duckdb.sql(query)
df_mediation_num_grouped

┌──────────────┬────────────┐
│ nb_mediation │ code_insee │
│    int64     │  varchar   │
├──────────────┼────────────┤
│            1 │ 01001      │
│            1 │ 01002      │
│            2 │ 01004      │
│            1 │ 01005      │
│            1 │ 01006      │
│            1 │ 01007      │
│            1 │ 01009      │
│            2 │ 01011      │
│            1 │ 01014      │
│            1 │ 01017      │
│            · │   ·        │
│            · │   ·        │
│            · │   ·        │
│            2 │ 97607      │
│            1 │ 97608      │
│            1 │ 97609      │
│            1 │ 97610      │
│            4 │ 97611      │
│            1 │ 97612      │
│            4 │ 97614      │
│            3 │ 97616      │
│            2 │ 97617      │
│            6 │ 97801      │
├──────────────┴────────────┤
│   6970 rows (20 shown)    │
└───────────────────────────┘

In [123]:
query = """ 
SELECT 
    dept_epci as dept_id,
    siren as epci_id,
    epci_nom as epci_lib,
    'i095' as id_indicator,
    round(10000 * sum(dmn.nb_mediation) / df_epci.total_pop_mun, 2) AS mediation_per_10k_habs ,
    '2025' as annee
FROM df_epci
LEFT JOIN df_mediation_num_grouped as dmn
ON dmn.code_insee = df_epci.code_insee
GROUP BY dept_epci, siren, epci_nom, total_pop_mun
ORDER BY dept_epci, siren
"""

df_mediation_num_final = duckdb.sql(query)
df_mediation_num_final

┌─────────┬───────────┬────────────────────────────────────────────────────────────┬──────────────┬────────────────────────┬─────────┐
│ dept_id │  epci_id  │                          epci_lib                          │ id_indicator │ mediation_per_10k_habs │  annee  │
│ varchar │   int64   │                          varchar                           │   varchar    │         double         │ varchar │
├─────────┼───────────┼────────────────────────────────────────────────────────────┼──────────────┼────────────────────────┼─────────┤
│ 01      │ 200029999 │ CC Rives de l'Ain - Pays du Cerdon                         │ i095         │                   8.74 │ 2025    │
│ 01      │ 200040350 │ CC Bugey Sud                                               │ i095         │                   5.28 │ 2025    │
│ 01      │ 200042497 │ CC Dombes Saône Vallée                                     │ i095         │                    4.4 │ 2025    │
│ 01      │ 200042935 │ CA Haut-Bugey Agglomération    

In [125]:
df_mediation_num_final.write_csv("../data/data_medi_num/processed/i095_mediation_numerique.csv")